# HCDE: train, evaluate and export the assistance-calibrated model

**Before you run:** Settings → Accelerator **GPU T4** (x1 or x2: only one GPU is used), Internet **on**.
Add-ons → Secrets, then tick each secret so this notebook can read it:

| Secret | Value |
| --- | --- |
| `GITHUB_TOKEN` | only if the AnchorAI repo is private |
| `TEACHER_BASE_URL` | `https://api.groq.com/openai/v1` |
| `TEACHER_API_KEY` | your Groq key(s), comma-separated |
| `TEACHER_MODEL` | `openai/gpt-oss-120b` |
| `JUDGE_MODEL` | `qwen/qwen3.8-27b` |
| `JUDGE_EXTRA` | `{"reasoning_effort": "none"}` |
| `JUDGE_MAX_TOKENS` | `600` |

Then **Run All**. Training takes about 1-1.5 hours; evaluation with the judge about another hour.
Every step skips itself if its output already exists, so after a crash just fix and Run All again.

In [ ]:
import os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU only: two make the Trainer split the model and run out of memory
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ""

BRANCH = "dev/dulanja"                     # the branch that holds train/val/test.jsonl
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"    # fallback if out of memory: Qwen/Qwen2.5-3B-Instruct
RUN = "/kaggle/working/runs/hcde-qwen7b"
REPO = "/kaggle/working/AnchorAI"

token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/DulanjaR/AnchorAI.git" if token else "https://github.com/DulanjaR/AnchorAI.git"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)   # always use the latest code
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
os.chdir(REPO)

loaded = []
for k in ["TEACHER_BASE_URL", "TEACHER_API_KEY", "TEACHER_MODEL", "TEACHER_EXTRA",
          "JUDGE_MODEL", "JUDGE_EXTRA", "JUDGE_MAX_TOKENS"]:
    if secret(k):
        os.environ[k] = secret(k)
        loaded.append(k)
print("secrets loaded:", loaded or "none (evaluation will run without the judge)")
!git log --oneline -1
!wc -l components/c4_hcde/training/data/dataset/train.jsonl components/c4_hcde/training/data/dataset/val.jsonl components/c4_hcde/training/data/dataset/test.jsonl

In [ ]:
!pip install -q -U peft bitsandbytes accelerate
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import transformers, peft, torch
print(transformers.__version__, peft.__version__, torch.__version__, "| GPUs visible:", torch.cuda.device_count())

## 1. Train (QLoRA)
Expect `0/81` steps. Loss is printed every few steps and should fall from about 2 to well under 1.

In [ ]:
if os.path.exists(f"{RUN}/adapter/adapter_config.json"):
    print("adapter already trained, skipping (delete", RUN, "to retrain)")
else:
    !python -m components.c4_hcde.training.train_qlora --base-model {BASE_MODEL} --out {RUN}
!tail -c 600 {RUN}/training_meta.json 2>/dev/null; echo

## 2. Quick look: one held-out task in every mode

In [ ]:
import json, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from components.c4_hcde.training.tiers import MODES, control_system_prompt

tok = AutoTokenizer.from_pretrained(f"{RUN}/adapter")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0}), f"{RUN}/adapter")
task = json.loads(open("components/c4_hcde/training/data/dataset/test.jsonl").readline())["messages"][1]["content"]
print("TASK:", task, "\n")
for mode in MODES:
    msgs = [{"role": "system", "content": control_system_prompt(mode)}, {"role": "user", "content": task}]
    text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=400, do_sample=False)
    print(f"===== {mode} =====\n{tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)}\n")
# free the GPU for the next steps, which load the model in a separate process
del model; gc.collect(); torch.cuda.empty_cache()

## 3. Evaluate on the 16 held-out IT tasks (3 conditions)
base model + short tag, base model + full rubric, fine-tuned model + short tag. The judge adds disclosure (SDS) scores.

In [ ]:
judge = "--judge" if os.environ.get("TEACHER_BASE_URL") else ""
if os.path.exists(f"{RUN}/eval/summary.json"):
    print("already evaluated, skipping")
else:
    !python -m components.c4_hcde.training.evaluate --base-model {BASE_MODEL} --adapter {RUN}/adapter --out {RUN}/eval {judge}
!cat {RUN}/eval/summary.json 2>/dev/null

## 4. Transfer test: 20 non-IT tasks the model never saw

In [ ]:
if os.path.exists(f"{RUN}/eval-transfer/summary.json"):
    print("already evaluated, skipping")
else:
    !python -m components.c4_hcde.training.evaluate --base-model {BASE_MODEL} --adapter {RUN}/adapter \
        --test components/c4_hcde/training/data/dataset/transfer_raw.jsonl \
        --elements components/c4_hcde/training/data/dataset/transfer_elements.json \
        --out {RUN}/eval-transfer {judge}
!cat {RUN}/eval-transfer/summary.json 2>/dev/null

## 5. Export to GGUF for Ollama on your Mac
If this runs out of memory: Run → Restart, then run the first cell and this one.

In [ ]:
!apt-get -qq install -y cmake > /dev/null
!python -m components.c4_hcde.training.export_gguf --base-model {BASE_MODEL} --adapter {RUN}/adapter --out /kaggle/working/gguf

## 6. Package results for download
From the Output panel download `hcde-results.zip` and the files in `gguf/` (the `.gguf` model and the `Modelfile`).

In [ ]:
!cd /kaggle/working && zip -qr hcde-results.zip runs/hcde-qwen7b/adapter runs/hcde-qwen7b/training_meta.json runs/hcde-qwen7b/eval runs/hcde-qwen7b/eval-transfer
!ls -lh /kaggle/working /kaggle/working/gguf